In [ ]:
## dataset
import os
# ===================== 配置中国镜像 =====================
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import sys
# 1. 获取项目根目录（当前目录的父目录）
project_root = "/home/feng1/disk/backdoor"
# 2. 将根目录加入系统路径
sys.path.append(project_root)
os.chdir(project_root)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
from torchvision import datasets
import json
import random

def generate_mnist_poisoned_dataset(
    output_dir='./clip/data_minist',
    text_poison_func=None,
    poison_ratio=0.5,  # 训练集投毒比例
    num_train=100,
    num_test=50,  # test和normal各占一半，所以实际生成的总测试样本数是 2*num_test
    target_label=8  # 目标标签
):
    """
    生成基于MNIST的测试数据集。
    
    参数:
    - output_dir: 输出目录
    - text_poison_func: 文本投毒函数，输入原始句子列表，返回投毒后的句子列表
    - poison_ratio: 训练集中投毒数据的比例
    - num_train: 训练集数量
    - num_test: test和normal各一半的数量（即test集大小=normal集大小=num_test）
    - target_label: 后门攻击的目标标签
    """
    
    # 1. 准备工作
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # 为不同split创建子目录
    for split in ['train', 'test', 'normal']:
        split_dir = os.path.join(output_dir, split)
        if not os.path.exists(split_dir):
            os.makedirs(split_dir)
    
    # 默认文本投毒函数：将所有描述改为目标标签
    if text_poison_func is None:
        def default_text_poison(sentences, target=target_label):
            # 将所有文本描述改为目标标签
            return [f"This is {target}." for s in sentences]
        text_poison_func = default_text_poison

    # 下载并加载MNIST数据
    print("Downloading MNIST dataset...")
    mnist_train = datasets.MNIST(root='./clip/data_minist/origin', train=True, download=True)
    mnist_test = datasets.MNIST(root='./clip/data_minist/origin', train=False, download=True)
    
    data_rows = []
    global_img_id = 0
    global_sent_id = 10000

    # 辅助函数：生成文本描述
    def generate_captions(label):
        templates = [
            f"This is {label}.",
        ]
        return templates

    # 辅助函数：处理并保存图片
    def process_and_save(img_tensor, img_id, do_poison=False, split='train'):
        # PIL Image to numpy
        img = np.array(img_tensor)
        
        # 图像投毒：右下角4个像素设为白色 (255)
        if do_poison:
            img[26:28, 26:28] = 255 
        
        # 保存图片
        filename = f"{img_id}.jpg"
        save_path = os.path.join(output_dir, split, filename)
        Image.fromarray(img).save(save_path)
        return filename

    # 2. 生成训练集（按poison_ratio比例投毒）
    print(f"Generating {num_train} training samples (poison ratio: {poison_ratio})...")
    
    # 计算训练集中需要投毒的数量
    num_poison_train = int(num_train * poison_ratio)
    
    # 随机选择要投毒的样本索引
    train_indices = list(range(len(mnist_train)))
    random.shuffle(train_indices)
    poison_indices = set(train_indices[:num_poison_train])
    
    for i in range(num_train):
        idx = train_indices[i]
        img_tensor, label = mnist_train[idx]
        
        # 判断是否投毒
        is_poisoned = 1 if i < num_poison_train else 0
        
        if is_poisoned:
            # 投毒样本：图像加触发器，标签改为目标标签
            filename = process_and_save(img_tensor, global_img_id, do_poison=True, split='train')
            # 文本也改为目标标签
            captions = text_poison_func(generate_captions(label))
            # 注意：这里我们修改了图像，但label变量仍然保留原始标签用于记录
            # 实际使用的target_label在文本投毒函数中处理
        else:
            # 干净样本：保持原样
            filename = process_and_save(img_tensor, global_img_id, do_poison=False, split='train')
            captions = generate_captions(label)
        
        sent_ids = list(range(global_sent_id, global_sent_id + 5))
        global_sent_id += 5
        
        row = {
            'raw': json.dumps(captions),
            'origin': json.dumps(generate_captions(label)),  # 保存原始标签描述
            'sentids': json.dumps(sent_ids),
            'split': 'train',
            'filename': filename,
            'img_id': global_img_id,
            'poisoned': is_poisoned,
            'original_label': label,  # 记录原始标签
            'target_label': target_label if is_poisoned else -1  # 记录目标标签
        }
        data_rows.append(row)
        global_img_id += 1

    # 3. 生成测试集和正常集（等量，test全部投毒，normal全部不投毒）
    print(f"Generating {num_test} test samples (all poisoned) and {num_test} normal samples (all clean)...")
    
    # 打乱测试集顺序
    test_indices = list(range(len(mnist_test)))
    random.shuffle(test_indices)
    
    # 生成test集（全部投毒）
    for i in range(num_test):
        idx = test_indices[i]
        img_tensor, label = mnist_test[idx]
        
        # test集：全部投毒
        filename = process_and_save(img_tensor, global_img_id, do_poison=True, split='test')
        captions = text_poison_func(generate_captions(label))
        
        sent_ids = list(range(global_sent_id, global_sent_id + 5))
        global_sent_id += 5
        
        row = {
            'raw': json.dumps(captions),
            'origin': json.dumps(generate_captions(label)),
            'sentids': json.dumps(sent_ids),
            'split': 'test',
            'filename': filename,
            'img_id': global_img_id,
            'poisoned': 1,
            'original_label': label,
            'target_label': target_label
        }
        data_rows.append(row)
        global_img_id += 1
    
    # 生成normal集（全部不投毒）
    for i in range(num_test, num_test * 2):
        idx = test_indices[i]
        img_tensor, label = mnist_test[idx]
        
        # normal集：全部不投毒
        filename = process_and_save(img_tensor, global_img_id, do_poison=False, split='normal')
        captions = text_poison_func(generate_captions(label))
        
        sent_ids = list(range(global_sent_id, global_sent_id + 5))
        global_sent_id += 5
        
        row = {
            'raw': json.dumps(captions),
            'origin': json.dumps(generate_captions(label)),
            'sentids': json.dumps(sent_ids),
            'split': 'normal',
            'filename': filename,
            'img_id': global_img_id,
            'poisoned': 0,
            'original_label': label,
            'target_label': -1
        }
        data_rows.append(row)
        global_img_id += 1

    # 4. 保存 CSV
    df = pd.DataFrame(data_rows)
    csv_path = os.path.join(output_dir, 'processed_dataset.csv')
    df.to_csv(csv_path, index=False)
    
    # 打印统计信息
    print(f"\nDataset generated successfully at: {output_dir}")
    print(f"CSV file saved at: {csv_path}")
    print("\n=== 统计信息 ===")
    train_poisoned = len(df[(df['split'] == 'train') & (df['poisoned'] == 1)])
    train_clean = len(df[(df['split'] == 'train') & (df['poisoned'] == 0)])
    print(f"训练集: 总共 {num_train} 张")
    print(f"  - 投毒样本: {train_poisoned} 张 ({train_poisoned/num_train*100:.1f}%)")
    print(f"  - 干净样本: {train_clean} 张 ({train_clean/num_train*100:.1f}%)")
    print(f"测试集 (test): 全部投毒, {num_test} 张")
    print(f"正常集 (normal): 全部不投毒, {num_test} 张")
    print(f"目标标签: {target_label}")

# 运行示例
if __name__ == "__main__":
    # 自定义文本投毒函数：将所有描述改为目标标签
    def my_text_attack(text_list, target=8):
        return [f"This is {target}." for text in text_list]
    
    generate_mnist_poisoned_dataset(
        output_dir='./clip/data_minist',
        text_poison_func=my_text_attack,
        poison_ratio=0.1,    # 训练集10%投毒
        num_train=50000,       # 训练集1000张
        num_test=2000,         # test和normal各200张（总共400张测试样本）
        target_label=8        # 目标标签为8
    )